In [8]:
# To run unix commands
import os
import sys
import json 

# Data analysis
import numpy as np
import pandas as pd
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Parameters

In [9]:
# Freqtrade directory
freqtrade_directory = '/home/jdpg/Documents/crypto_bot'
# User data folder
user_data_folder = f'user_data'
# Code folder (where the scripts are)
code_folder = 'code'
# Backtest folder
btest_folder = 'backtest_results'
# Pairlists file
pairlists_file = 'pairlists.txt'    # Ordered by volume
# Strategies file 
strategies_file = 'best_strategies.txt'  # Ordered by performance (third party calculated)

# How many strategies and pairlists we want to backtest
n_strategies = 20
n_pairlists = 10 

# Exchange
exchange = 'binance'

# Initial and Final date in format YYYMMDD
date_start = '20220401'#'20241123'## BTC peak after crash from sept 2021
date_end = '20230930'#'20250215'# # Valley Before the rally

# Loads the pairlists
with open(f'{freqtrade_directory}/{user_data_folder}/{pairlists_file}', 'r') as f:
    pairlists = f.read().splitlines()
# Loads the strategies
with open(f'{freqtrade_directory}/{user_data_folder}/{strategies_file}', 'r') as f:
    strategies = f.read().splitlines()

# Selects the top n strategies and pairlists
pairlists = pairlists[:n_pairlists]
strategies = strategies[:n_strategies]

'''
    Given the parameters, generates the script to run the backtest
    
    Args:
        - strategies: list of strategies to backtest
        - starting_balance: starting balance for the backtest
        - max_open_trades: maximum number of open trades
        - enable_position_stacking: whether to allow buying the same asset multiple times
        - stake_amount: amount to stake in each trade. Plain value or 'unlimited' for percentage calculated according to balance and max_open_trades
        - start_date: starting date for the backtest in format YYYMMDD
        - end_date: final date for the backtest in format YYYMMDD
        - timeframe: timeframe for the backtest (e.g. 1m, 1h, 1d)
        - data_breakdown: data breakdown for the backtest (e.g. day, week, month)
        - export_fname: filename to export the backtest results
    Returns:
        - backtest_script: string with the backtest script to run in console.
'''
def get_backtest_script(strategies = [],
                        pairs = [],
                        starting_balance = 1000,
                        max_open_trades = 5,
                        enable_position_stacking = False,
                        stake_amount = 0.1,
                        start_date = '',
                        end_date = '',
                        timeframe = '1m',
                        data_breakdown = 'day',
                        export_fname = 'backtest_results'):

    # Gets strategies list to string 
    strategies_str = ' '.join(strategies)
    # Gets the list of pairs to string
    pairs_str = ' '.join(pairs)
    # Gest the script
    script  = ('freqtrade backtesting '
               # Strategy setup
               '--strategy-list {strategies_str} '
               # Pairlist setup
               '--pairs {pairs_str} '
               # Trading params
               '--starting-balance {starting_balance} --max-open-trades {max_open_trades} {enable_position_stacking} --stake-amount {stake_amount} '
               # Time params
               '--timerange {start_date}-{end_date} --timeframe {timeframe} --breakdown {data_breakdown} '
               # Export params
               '--export trades --export-filename={export_fname}.json')
    
    # Returns the script
    return eval(f'f"""{script}"""')

# Config file to populate backtest script
config =   {# Strategy setup
            'strategies': strategies,
            # Pairlist setup
            'pairs': pairlists,
            # Initial balance
            'starting_balance': 1000,
            # Trading params
            'max_open_trades': 10,
            'enable_position_stacking': '--enable-position-stacking',
            'stake_amount': "unlimited",
            # Time params
            'start_date': date_start,
            'end_date': date_end,
            'timeframe': '1h',
            'data_breakdown': 'day',
            # Export params
            'export_fname': f'{user_data_folder}/{btest_folder}/stress_backtest'}

# prints the formatted script
print(get_backtest_script(**config))

freqtrade backtesting --strategy-list ichiV1 HarmonicDivergence Babico_SMA5xBBmid TheForce Ichimoku_v37 keltnerchannel WaveTrendStra EMA520015_V17 Slowbro NFINextMOHO BigZ04HO2 HyperStra_GSN_SMAOnly HyperStra_SMAOnly CombinedBinHClucAndMADV6 CombinedBinHClucAndMADV9 CombinedBinHClucAndMADV9 BigZ0407HO hlhb CombinedBinHClucAndMADV5 BinClucMad --pairs BTC/USDC ETH/USDC XRP/USDC SOL/USDC FDUSD/USDC DOGE/USDC ADA/USDC SUI/USDC PEPE/USDC EUR/USDC --starting-balance 1000 --max-open-trades 10 --enable-position-stacking --stake-amount unlimited --timerange 20220401-20230930 --timeframe 1h --breakdown day --export trades --export-filename=user_data/backtest_results/stress_backtest.json


In [ ]:
# The timeframes to test
timeframes_to_test = ['5m', '15m', '1h', '4h', '1d'][::-1]# Sort them in descending order

# Backtest results timeframes to filenames
backtest_results_paths = {}

# Gets current directory
cwd = os.getcwd()

if cwd != freqtrade_directory:
    # Changes directory to the freqtrade folder
    os.chdir(freqtrade_directory)

# Iterates over the timeframes
for timeframe in timeframes_to_test:
    # Updates the config
    config['timeframe'] = timeframe
    config['export_fname'] = f'{user_data_folder}/{btest_folder}/stress_backtest_{timeframe}.json'
    # Generates the script
    script = get_backtest_script(**config)
    
    # Runs the script
    print(f"Running backtest for timeframe {timeframe}")
    os.system(script)
    
    # Loads the file with the latest backtest results
    latest_backtest_json = f'{freqtrade_directory}/{user_data_folder}/{btest_folder}/.last_result.json'
    # Opens the file to obtain the latest backtest results
    with open(latest_backtest_json, 'r') as f:
        latest_backtest = json.load(f)
    # Saves the backtest results
    backtest_results_paths[timeframe] = latest_backtest['latest_backtest']

# Saves the backtest results paths
with open(f'{freqtrade_directory}/{user_data_folder}/{btest_folder}/backtest_results_paths.json', 'w') as f:
    json.dump(backtest_results_paths, f)

Running backtest for timeframe 1d


2025-02-24 19:28:41,721 - freqtrade - INFO - freqtrade 2025.2-dev-f14c4ebcc
2025-02-24 19:28:41,833 - numexpr.utils - INFO - Note: NumExpr detected 24 cores
but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
2025-02-24 19:28:41,834 - numexpr.utils - INFO - NumExpr defaulting to 16 
threads.
2025-02-24 19:28:42,181 - freqtrade.configuration.load_config - INFO - Using 
config: user_data/config.json ...
2025-02-24 19:28:42,181 - freqtrade.loggers - INFO - Enabling colorized output.
2025-02-24 19:28:42,182 - freqtrade.loggers - INFO - Verbosity set to 0
2025-02-24 19:28:42,182 - freqtrade.configuration.configuration - INFO - 
Parameter -i/--timeframe detected ... Using timeframe: 1d ...
2025-02-24 19:28:42,182 - freqtrade.configuration.configuration - INFO - 
Parameter --enable-position-stacking detected ...
2025-02-24 19:28:42,182 - freqtrade.configuration.configuration - INFO - 
Parameter --max-open-trades detected, overriding max_open_trades to: 10 ...
2025-02-24 19:28:42

In [18]:
backtest_results

{'1d': 'stable_backtest-2025-02-23_12-33-42.zip'}

# Data Loading

In [4]:
from freqtrade.data.btanalysis import load_backtest_data, load_backtest_stats
# Sample file
btest_file = f'../{btest_folder}/stress_backtest-2025-02-22_22-06-05.json'
btest_file

'../backtest_results/stress_backtest-2025-02-22_22-06-05.json'

In [5]:
# Plotting equity line (starting with 0 on day 1 and adding daily profit for each backtested day)
from freqtrade.configuration import Configuration
from freqtrade.data.btanalysis import load_backtest_data, load_backtest_stats
import plotly.express as px
import pandas as pd

def get_daily_profit_balance1(backtest_path, strategy):
    """
    Returns a Dataframe with the fields (strategy, date, profit, balance and profit_percent) each day of the backtest
    -------------------------------------------------
    Parameters:
        backtest_path: path to backtest folder
        strategy: strategy name
    -------------------------------------------------
    Returns:
        Dataframe with the fields (strategy, date, profit, balance and profit_percent) each day of the backtest
    """
    # Loads backtest data
    stats = load_backtest_stats(backtest_path)
    # Filters strategy data
    strategy_stats = stats['strategy'][strategy]
    # Populates dates and profits
    dates = []
    profits = []
    for date_profit in strategy_stats['daily_profit']:
        dates.append(date_profit[0])
        profits.append(float(date_profit[1]))
    
    # Populates balance which is the starting balance + cumulative daily profit
    current_balance = config['starting_balance']
    balance = []
    for daily_profit in profits:
        current_balance += daily_profit
        balance.append(current_balance)
    
    # Populates profit_percent which is the daily profit divided by the balance the day before
    profit_percent = []
    for i in range(len(profits)):
        if i == 0:
            profit_percent.append(profits[0]/config['starting_balance'])
        else:
            profit_percent.append(profits[i]/balance[i-1])
    
    # Creates dataframe 
    df = pd.DataFrame({'strategy': [strategy]*len(dates), 'date': dates, 'profit': profits, 'balance': balance, 'profit_percent': profit_percent})

    return df

In [6]:
# Iterates over strategies creating a dataframe with the fields (strategy, date, profit, balance and profit_percent) each day of the backtest
summary_dfs = []
for strategy in strategies:
    try:
        summary_dfs.append(get_daily_profit_balance1(btest_file, strategy))
    except:
        print(f'{strategy} Strategy Defective file: {btest_file}')
        break

summary_df = pd.concat(summary_dfs)
print(summary_df)

        strategy        date     profit      balance  profit_percent
0         ichiV1  2022-04-01  12.732723  1012.732723        0.012733
1         ichiV1  2022-04-02  -0.005656  1012.727067       -0.000006
2         ichiV1  2022-04-03   0.000000  1012.727067        0.000000
3         ichiV1  2022-04-04  -0.394785  1012.332282       -0.000390
4         ichiV1  2022-04-05   0.000000  1012.332282        0.000000
..           ...         ...        ...          ...             ...
520  NFINextMOHO  2023-09-03   0.000000   418.171092        0.000000
521  NFINextMOHO  2023-09-04   0.000000   418.171092        0.000000
522  NFINextMOHO  2023-09-05   0.000000   418.171092        0.000000
523  NFINextMOHO  2023-09-06   0.000000   418.171092        0.000000
524  NFINextMOHO  2023-09-07   1.269465   419.440557        0.003036

[4803 rows x 5 columns]


# Plots & Data Analysis

In [8]:
# Plots Daily Balance for each strategy
fig = px.line(summary_df, x='date', y='balance', color='strategy', title='Daily Balance')
fig.show()
# Plots Daily Profit for each strategy
fig = px.line(summary_df, x='date', y='profit', color='strategy', title='Daily Profit')
fig.show()
# Plots Daily Profit Percent for each strategy
fig = px.line(summary_df, x='date', y='profit_percent', color='strategy', title='Daily Profit Percent')
fig.show()

In [30]:
# Calculates weekly rolling average of each field for each strategy
summary_df['week_profit_moving_average'] = summary_df.groupby('strategy')['profit'].transform(lambda x: x.rolling(window=7).mean())
summary_df['week_balance_moving_average'] = summary_df.groupby('strategy')['balance'].transform(lambda x: x.rolling(window=7).mean())
summary_df['week_profit_percent_moving_average'] = summary_df.groupby('strategy')['profit_percent'].transform(lambda x: x.rolling(window=7).apply(lambda x: np.exp(np.mean(np.log(1+x)))-1))
summary_df['week_profit_percent_moving_std'] = summary_df.groupby('strategy')['profit_percent'].transform(lambda x: x.rolling(window=7).apply(lambda x: np.exp(np.std(np.log(1+x)))-1))

In [31]:
# Selects only strategies with final balance > config['starting_balance']
strategies_filter = summary_df[summary_df.groupby(['strategy'])['balance'].transform('last') > config['starting_balance']].strategy.unique()
strategies_filter = strategies
# Filters strategy
summary_filtered_df = summary_df.loc[summary_df['strategy'].isin(strategies_filter)]

# Plots Average profit perfent and its variance as continuous error lines per week for each strategy
fig = px.line(summary_filtered_df, x='date', y='week_profit_percent_moving_average', color='strategy', title='Weekly Profit Percent Moving Average')
# Draws horizontal line at 0
fig.add_shape(type='line', 
              x0=summary_filtered_df['date'].min(), y0=0,
              x1=summary_filtered_df['date'].min(), y1=0,
              line=dict(color='black', width=2))
fig.show()

In [32]:
# Import graphic objects
import plotly.graph_objs as go

def line(error_y_mode=None, **kwargs):
    """Extension of `plotly.express.line` to use error bands."""
    ERROR_MODES = {'bar','band','bars','bands',None}
    if error_y_mode not in ERROR_MODES:
        raise ValueError(f"'error_y_mode' must be one of {ERROR_MODES}, received {repr(error_y_mode)}.")
    if error_y_mode in {'bar','bars',None}:
        fig = px.line(**kwargs)
    elif error_y_mode in {'band','bands'}:
        if 'error_y' not in kwargs:
            raise ValueError(f"If you provide argument 'error_y_mode' you must also provide 'error_y'.")
        figure_with_error_bars = px.line(**kwargs)
        fig = px.line(**{arg: val for arg,val in kwargs.items() if arg != 'error_y'})
        for data in figure_with_error_bars.data:
            x = list(data['x'])
            y_upper = list(data['y'] + data['error_y']['array'])
            y_lower = list(data['y'] - data['error_y']['array'])
            color = f"rgba({tuple(int(data['line']['color'].lstrip('#')[i:i+2], 16) for i in (0, 2, 4))},.3)".replace('((','(').replace('),',',').replace(' ','')
            fig.add_trace(
                go.Scatter(
                    x = x+x[::-1],
                    y = y_upper+y_lower[::-1],
                    fill = 'toself',
                    fillcolor = color,
                    line = dict(color = 'rgba(255,255,255,0)'),
                    hoverinfo = "skip",
                    showlegend = False,
                    legendgroup = data['legendgroup'],
                    xaxis = data['xaxis'],
                    yaxis = data['yaxis'],
                )
            )
        # Reorder data as said here: https://stackoverflow.com/a/66854398/8849755
        reordered_data = []
        for i in range(int(len(fig.data)/2)):
            reordered_data.append(fig.data[i+int(len(fig.data)/2)])
            reordered_data.append(fig.data[i])
        fig.data = tuple(reordered_data)
    return fig


summary_filtered_df['week_profit_percent_moving_std_half'] = summary_filtered_df['week_profit_percent_moving_std']/4
for strategy in strategies_filter:
    # Gets strat_df
    strat_df = summary_filtered_df.loc[summary_filtered_df['strategy'] == strategy]
    # Plots Average profit perfent and its variance as continuous error lines per week for each strategy
    fig = line( data_frame = strat_df,
                x = 'date',
                y = 'week_profit_percent_moving_average',
                error_y = 'week_profit_percent_moving_std_half',
                error_y_mode = 'band',
                color = 'strategy',
                title = f'[{strategy}] Moving Average of Profit Percent & Variance')
    fig.show()

In [33]:
# Loads backtest data
stats = load_backtest_stats(btest_file)

In [34]:
def get_trades_strategy_pair(backtest_path):
    """
    Returns a Dataframe with the fields (strategy, pair, stake_amount, amount, open_date, close_date, profit_ratio, profit abs, stop_loss_ratio, initial_stop_loss_ratio))  
    from the backtest
    -------------------------------------------------
    Parameters:
        backtest_path: path to backtest file
    -------------------------------------------------
    Returns:
        Dataframe with selected field
    """
    # Loads backtest data
    stats = load_backtest_stats(backtest_path)

    # Initializes dataframe
    trades_df = pd.DataFrame()

    # Iterates over strategies
    for strategy in stats['strategy']:
        # Continue if Pair has no trades
        if 'trades' not in stats['strategy'][strategy]:
            continue
        
        # Creates strategy dataframe
        strat_df = pd.DataFrame(stats['strategy'][strategy]['trades'])
        strat_df['strategy'] = strategy

        # Appends to trades_df
        trades_df = pd.concat([trades_df,strat_df])
    
    # Returns dataframe
    return trades_df


In [35]:
strategy_pair_trades = get_trades_strategy_pair(btest_file)

In [36]:
# Gets important columns
strategy_pair_trades = strategy_pair_trades[['strategy', 'pair', 'open_date', 'close_date', 'profit_ratio']]

# Initializes grouped dataframe
strategy_pair_trades_grouped = strategy_pair_trades[['strategy', 'pair']]
strategy_pair_trades_grouped.set_index(['strategy', 'pair'], inplace = True)

# Gets the average, std, count profit ratio per strategy pair
strategy_pair_trades_grouped['average_profit_ratio'] = strategy_pair_trades.groupby(['strategy', 'pair'])['profit_ratio'].agg(lambda x: np.exp(np.mean(np.log(1+x)))-1)
strategy_pair_trades_grouped['std_profit_ratio'] = strategy_pair_trades.groupby(['strategy', 'pair'])['profit_ratio'].agg(lambda x: np.exp(np.std(np.log(1+x)))-1)
strategy_pair_trades_grouped['count_profit_ratio'] = strategy_pair_trades.groupby(['strategy', 'pair'])['profit_ratio'].count()

# Gets only unique values 
strategy_pair_trades_grouped = strategy_pair_trades_grouped.drop_duplicates()

/tmp/ipykernel_47249/987694807.py:9: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_47249/987694807.py:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_47249/987694807.py:11: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [37]:
strategy_pair_trades_grouped = strategy_pair_trades_grouped.reset_index()
# Filters strategies
strategy_pair_trades_grouped = strategy_pair_trades_grouped.loc[strategy_pair_trades_grouped['strategy'].isin(strategies_filter)]

In [38]:
# Plots separated bar chart for average profit ratio with plotly. Horizontal
# Chart is more tall than wide                
fig = px.bar(   strategy_pair_trades_grouped, 
                y = 'pair',
                x = 'average_profit_ratio',
                color = 'strategy',
                title = 'Average Profit Ratio per Strategy Pair',
                orientation = 'h',
                barmode = 'group',
                width = 800,
                height = 1200)

fig.show()

In [39]:
# Plots separated bar chart for average profit ratio with plotly. Horizontal
# Chart is more tall than wide                
fig = px.bar(   strategy_pair_trades_grouped, 
                y = 'pair',
                x = 'average_profit_ratio',
                color = 'strategy',
                title = 'Average Profit Ratio per Strategy Pair',
                orientation = 'h',
                barmode = 'group',
                width = 800,
                height = 1200)

fig.show()

In [40]:
# Plots separated bar chart for count profit ratio with plotly. Horizontal
# Chart is more tall than wide                
fig = px.bar(   strategy_pair_trades_grouped, 
                y = 'pair',
                x = 'count_profit_ratio',
                color = 'strategy',
                title = 'Count Profit Ratio per Strategy Pair',
                orientation = 'h',
                barmode = 'group',
                width = 800,
                height = 1200)
fig.show()